# Matemáticas de la Inteligencia Artificial
## Sesión 6 — Probabilidad, información y clasificación probabilística

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CuentosCuanticos/matematicas-ia/blob/main/06_probabilidad_softmax/laboratorio.ipynb)

### Pregunta de la sesión
**¿Cómo pasa una red de producir números a expresar distribuciones de probabilidad y cómo aprende esas distribuciones a partir de datos?**

Trabajaremos solo con **NumPy** y **Matplotlib**. El itinerario será

\[\text{frecuencias}\to P(Y\mid X)\to\text{Bayes}\to\text{Bernoulli}\to\text{logit/sigmoide}\to\text{softmax}\to\text{cross-entropy}\to\text{clasificador probabilístico}.\]

> Ejecuta las celdas de arriba abajo. Los bloques `TODO` son pasos que debes completar. No usaremos todavía PyTorch ni `autograd`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=5, suppress=True)
rng = np.random.default_rng(42)

# 1. De frecuencias a probabilidades

Partimos de la tabla de conteos

\[\begin{array}{c|cc|c}&Y=0&Y=1&\text{total}\\\hline X=0&30&10&40\\X=1&15&45&60\\\hline\text{total}&45&55&100\end{array}\]

La conjunta es \(\widehat p(x,y)=N(x,y)/N\). Las marginales se obtienen sumando sobre la variable que dejamos de especificar. Después,

\[P(Y=y\mid X=x)=\frac{P(X=x,Y=y)}{P(X=x)}.\]


In [ ]:
conteos = np.array([[30,10],[15,45]], dtype=float)
N = conteos.sum()

# TODO: conjunta y marginales
p_xy = ...
p_x = ...
p_y = ...

# TODO: condicionadas P(Y|X=0) y P(Y|X=1)
p_y_x0 = ...
p_y_x1 = ...

print('p(x,y)=\n', p_xy)
print('p(x)=', p_x)
print('p(y)=', p_y)
print('P(Y|X=0)=', p_y_x0)
print('P(Y|X=1)=', p_y_x1)
# Debes obtener P(Y|X=0)=[0.75,0.25] y P(Y|X=1)=[0.25,0.75]

# 2. Bayes y las tasas base

Supón \(P(D)=0.05\), \(P(+\mid D)=0.90\) y \(P(+\mid\neg D)=0.10\). Queremos \(P(D\mid+)\), no \(P(+\mid D)\).

\[P(D\mid+)=\frac{P(+\mid D)P(D)}{P(+)}\]

con

\[P(+)=P(+\mid D)P(D)+P(+\mid\neg D)P(\neg D).\]


In [ ]:
p_D = 0.05
p_pos_D = 0.90
p_pos_noD = 0.10

# TODO
p_pos = ...
p_D_pos = ...
print('P(+) =', p_pos)
print('P(D|+) =', p_D_pos)

# Verificación con una población de 10 000
poblacion = 10_000
n_D = ...
n_noD = ...
n_pos_D = ...
n_pos_noD = ...
print('posterior por conteos =', n_pos_D/(n_pos_D+n_pos_noD))

### Preguntas de interpretación
1. Explica por qué \(P(+\mid D)\neq P(D\mid+)\).
2. ¿Qué papel desempeña el prior \(P(D)\)?
3. ¿Qué ocurriría con el posterior si el fenómeno fuese mucho más frecuente, manteniendo el mismo test?


# 3. Bernoulli, logit y sigmoide

Para \(Y\in\{0,1\}\),
\[P(Y=y\mid p)=p^y(1-p)^{1-y}.\]
El estimador de máxima verosimilitud es \(\widehat p_{ML}=N^{-1}\sum_n y_n\). Para hacer depender la probabilidad de una entrada usamos
\[z=\log\frac{p}{1-p},\qquad p=\sigma(z)=\frac{1}{1+e^{-z}}.\]


In [ ]:
y_bern = np.array([1,0,1,1,0,1,1,1,0,1])
# TODO: MLE de p
p_mle = ...
print('p_ML =', p_mle)

def sigmoid(z):
    # TODO
    return ...

z = np.linspace(-8,8,400)
plt.figure(figsize=(7,4))
plt.plot(z, sigmoid(z))
plt.axhline(.5, linestyle='--')
plt.axvline(0, linestyle='--')
plt.xlabel('logit z'); plt.ylabel('probabilidad'); plt.grid(alpha=.25); plt.show()

# 4. Softmax y cross-entropy

Para \(K\) clases,
\[p_i=\frac{e^{z_i}}{\sum_j e^{z_j}},\qquad \log\frac{p_i}{p_j}=z_i-z_j.\]
Softmax no cambia si sumamos la misma constante a todos los logits. Por estabilidad numérica restaremos siempre el máximo.

Para una etiqueta one-hot,
\[L=-\sum_i y_i\log p_i=-\log p_c,\]
y el resultado crucial es
\[\nabla_{\mathbf z}L=\mathbf p-\mathbf y.\]


In [ ]:
def softmax(z):
    z = np.asarray(z)
    # TODO: restar máximo, exponentiar y normalizar
    m = ...
    ez = ...
    return ...

def one_hot(y, K):
    Y = np.zeros((len(y),K))
    Y[np.arange(len(y)),y] = 1.0
    return Y

def cross_entropy(P,Y):
    P = np.clip(P,1e-12,1.0)
    # TODO
    return ...

z_demo = np.array([1000.,1001.,1002.])
print('softmax estable =', softmax(z_demo))

# 5. Clasificador softmax desde cero

Construiremos tres nubes de puntos y un modelo lineal multiclase
\[Z=XW+b,\qquad P=\operatorname{softmax}(Z).\]
Con pérdida media,
\[dZ=\frac{P-Y}{N},\qquad \nabla_WL=X^TdZ,\qquad \nabla_bL=\sum_n dZ_n.\]


In [ ]:
def generar_datos(seed=7, n=80):
    rg=np.random.default_rng(seed)
    centros=np.array([[-1.8,-0.8],[1.6,-0.7],[0.0,1.7]])
    Xs=[]; ys=[]
    for k,c in enumerate(centros):
        Xs.append(rg.normal(c,[0.9,0.8],size=(n,2)))
        ys.append(np.full(n,k,dtype=int))
    X=np.vstack(Xs); y=np.concatenate(ys)
    p=rg.permutation(len(y)); return X[p],y[p]

X,y = generar_datos(); Y=one_hot(y,3)
W=rng.normal(0,.05,size=(2,3)); b=np.zeros(3)
eta=.15; historial=[]

for epoca in range(800):
    # TODO: forward
    Z = ...
    P = ...
    L = ...
    # TODO: gradientes
    dZ = ...
    gW = ...
    gb = ...
    # TODO: actualización
    W = ...
    b = ...
    historial.append(L)

print('L inicial=', historial[0], 'L final=', historial[-1])
plt.figure(figsize=(7,4)); plt.plot(historial); plt.xlabel('época'); plt.ylabel('cross-entropy'); plt.grid(alpha=.25); plt.show()

# TODO: distribución final y clasificación dura
P_final = ...
y_pred = ...
print('fracción de aciertos =', np.mean(y_pred==y))
for n in range(5): print('y=',y[n],'p=',P_final[n])

# 6. Entropía, KL y calibración

\[H(p)=-\sum_i p_i\log p_i,\qquad D_{KL}(p\|q)=\sum_i p_i\log\frac{p_i}{q_i}.\]
Comprueba numéricamente \(H(p,q)=H(p)+D_{KL}(p\|q)\). Después recuerda: una probabilidad de softmax es **probabilidad del modelo**, no certeza factual. La calibración compara confianza predicha con frecuencia empírica de aciertos.


In [ ]:
def entropia(p):
    p=np.asarray(p,float); m=p>0
    # TODO
    return ...

def kl(p,q):
    p=np.asarray(p,float); q=np.asarray(q,float); m=p>0
    # TODO
    return ...

p=np.array([.7,.2,.1]); q=np.array([.65,.25,.10])
H_pq = -np.sum(p*np.log(q))
print('H(p,q)=',H_pq)
print('H(p)+KL=', entropia(p)+kl(p,q))

# Problema final — De Bayes a un clasificador probabilístico

Una estación registra eventos de tres clases \(Y\in\{0,1,2\}\). Cada evento tiene dos variables continuas \(X=(x_1,x_2)\) y una alarma binaria \(A\in\{0,1\}\).

Debes integrar toda la sesión:

1. estima \(P(Y=k)\) y \(P(A=1\mid Y=k)\);
2. calcula \(P(A=1)\) por frecuencias y por probabilidad total;
3. usa Bayes para obtener \(P(Y=2\mid A=1)\) y compruébalo por conteos;
4. ignora después \(A\) y entrena con \(X\) un clasificador softmax lineal desde cero;
5. representa su curva de pérdida y examina las distribuciones en los puntos \((-1.5,-0.7)\), \((0,0.2)\) y \((0.1,1.8)\);
6. explica cuál es más ambiguo, qué información pierde `argmax`, por qué un `0.92` no es certeza factual y qué parte del ejercicio es Bayes frente a máxima verosimilitud.

La solución completa está reservada al profesorado.


In [ ]:
def generar_problema_final(seed=123):
    rg=np.random.default_rng(seed)
    n=[120,80,40]
    centros=np.array([[-1.6,-0.7],[1.5,-0.5],[0.1,1.8]])
    escalas=np.array([[.95,.75],[.85,.90],[.80,.80]])
    pA=np.array([.15,.55,.85])
    Xs=[]; ys=[]; As=[]
    for k in range(3):
        Xs.append(rg.normal(centros[k],escalas[k],size=(n[k],2)))
        ys.append(np.full(n[k],k,dtype=int))
        As.append(rg.binomial(1,pA[k],size=n[k]))
    X=np.vstack(Xs); y=np.concatenate(ys); A=np.concatenate(As)
    p=rg.permutation(len(y)); return X[p],y[p],A[p]

Xf,yf,Af=generar_problema_final(); Yf=one_hot(yf,3)
print('conteos por clase=',np.bincount(yf),'alarmas=',Af.sum())

# PARTE A — TODO
prior = ...
pA_clase = ...
pA = ...
posterior_y2 = ...
posterior_y2_conteos = ...
print('prior=',prior)
print('P(A=1|Y=k)=',pA_clase)
print('P(Y=2|A=1)=',posterior_y2,'comprobación=',posterior_y2_conteos)

# PARTE B — TODO: entrena 1000 épocas con eta=0.12
rgf=np.random.default_rng(2026)
Wf=...; bf=...; historial_final=[]
for epoca in range(1000):
    Zf=...; Pf=...; Lf=...
    dZf=...; gWf=...; gbf=...
    Wf=...; bf=...
    historial_final.append(Lf)

# PARTE C — TODO
puntos=np.array([[-1.5,-0.7],[0.,0.2],[0.1,1.8]])
probs=...
for x,pr in zip(puntos,probs): print(x,'->',pr)